In [1]:
import pandas as pd
import heapq

# Load cleaned dataset
df = pd.read_csv("../data/processed/edges_clean.csv")

# Build graph (adjacency list)
graph = {}
for _, row in df.iterrows():
    graph.setdefault(int(row["from_node"]), {})[int(row["to_node"])] = int(row["cost"])

graph


{1: {2: 15, 5: 19, 6: 11},
 2: {3: 2, 6: 1, 7: 12, 8: 10},
 3: {1: 16, 2: 15, 5: 3},
 4: {1: 14, 3: 2, 7: 8, 8: 14},
 5: {1: 18, 3: 6, 4: 4, 6: 18},
 6: {4: 8},
 7: {4: 15, 5: 7, 8: 8},
 8: {1: 11, 2: 8, 5: 7}}

In [2]:
def dijkstra(graph, start):
    # Initialize distances
    dist = {node: float("inf") for node in graph}
    dist[start] = 0

    # Priority queue (min-heap)
    pq = [(0, start)]

    while pq:
        current_dist, current_node = heapq.heappop(pq)

        # Skip if we already found a better path
        if current_dist > dist[current_node]:
            continue

        # Explore neighbors
        for neighbor, cost in graph.get(current_node, {}).items():
            new_dist = current_dist + cost

            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                heapq.heappush(pq, (new_dist, neighbor))

    return dist


In [3]:
start_node = list(graph.keys())[0]

distances = dijkstra(graph, start_node)

distances


{1: 0, 2: 15, 3: 17, 4: 19, 5: 19, 6: 11, 7: 27, 8: 25}

In [4]:
import heapq

def dijkstra_with_path(graph, start):
    dist = {node: float("inf") for node in graph}
    prev = {node: None for node in graph}

    dist[start] = 0
    pq = [(0, start)]

    while pq:
        current_dist, current_node = heapq.heappop(pq)

        if current_dist > dist[current_node]:
            continue

        for neighbor, cost in graph.get(current_node, {}).items():
            new_dist = current_dist + cost

            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                prev[neighbor] = current_node
                heapq.heappush(pq, (new_dist, neighbor))

    return dist, prev


In [5]:
def reconstruct_path(prev, start, target):
    path = []
    current = target

    while current is not None:
        path.append(current)
        current = prev[current]

    path.reverse()

    if path[0] == start:
        return path
    else:
        return []


In [6]:
start_node = 1
target_node = 5

dist, prev = dijkstra_with_path(graph, start_node)

path = reconstruct_path(prev, start_node, target_node)

print("Shortest path:", path)
print("Shortest cost:", dist[target_node])


Shortest path: [1, 5]
Shortest cost: 19
